In [0]:
df1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/",inferSchema=True).toDF("customer_id","Name","age","location","plan_type")

df1.show(6)

In [0]:
df_usage=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage.show(10)

In [0]:

df_tower=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*",header=True,sep='|',inferSchema=True)
df_tower.show(6)


df_tower=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*",header=True,sep='|')
df_tower.show(6)

In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer_rgt")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage_rgt")

In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_rgt/region1")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_rgt/region2")

#Telecom Domain Read & Write Ops Assignment - Building Datalake & Lakehouse
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS telecom_catalog_assign;
CREATE SCHEMA IF NOT EXISTS telecom_catalog_assign.landing_zone;
CREATE VOLUME IF NOT EXISTS telecom_catalog_assign.landing_zone.landing_vol;


In [0]:

dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2")

dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/")


##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

##2. Filesystem operations
1. Write dbutils.fs code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:

customer_csv = ''' 
101,Arun,31,Chennai,PREPAID 
102,Meera,45,Bangalore,POSTPAID 
103,Irfan,29,Hyderabad,PREPAID 
104,Raj,52,Mumbai,POSTPAID 
105,,27,Delhi,PREPAID 
106,Sneha,abc,Pune,PREPAID '''

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_csv.csv",customer_csv,True)


tower_logs_region1 = '''
event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12 '''

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_region1.csv",tower_logs_region1,True)

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_region2.csv",tower_logs_region1,True)



In [0]:

tower_logs_region1 = '''
event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12 '''

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_region1.csv",tower_logs_region1,True)

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_region2.csv",tower_logs_region1,True)

In [0]:
usage_csv = '''
customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20 
102\t120\t4000\t5 
103\t540\t600\t52 
104\t45\t200\t2 
105\t0\t0\t0 '''

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_tsv.csv",usage_csv,True)

In [0]:
tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp 5001|101|TWR01|-80|2025-01-10 10:21:54 5004|104|TWR05|-75|2025-01-10 11:01:12 '''

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_region1.csv",tower_logs_region1,True)

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs_region2.csv",tower_logs_region1,True)


In [0]:
dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer")
dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage")
dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")

##3. Spark Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
df2_all_tower=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/",header=True,inferSchema=True,recursiveFileLookup=True,
pathGlobFilter="tower*",sep='|')
df2_all_tower.display()

In [0]:
df2_all_tower=spark.read.format("csv").option("header","True").option("delimiter","|").option("recursiveFileLookup","True").option("pathGlobFilter","*.csv")\
.load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")
df2_all_tower.display()

df2_all=spark.read.format("csv").option("header","True").option("delimiter",",").option("recursiveFileLookup","True").option("pathGlobFilter","*.csv")\
.load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol")
df2_all.display()

In [0]:

df3_all_log=spark.read.option("recursiveFileLookup","true").csv(path=[
"/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",
"/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/",
"/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage"],header=True)
df3_all_log.display()

In [0]:
df1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/").toDF("customer_id","Name","age","location","plan_type")

df1.show(5)

##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>

In [0]:
df_cust_false=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",header=False,inferSchema=False)
df_cust_false.show(5)

df_cust_true=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",header=True,inferSchema=True)
df_cust_true.show(5)

##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
df2 = spark.read.csv(
    path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/",
    header=True,
    inferSchema=True)

df2.printSchema()
df2.show(10)

In [0]:
datastruct="cust_id int,name string,age int,city string,plan_type string"
df2=spark.read.schema(datastruct).csv(
    path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer")
df2.printSchema()
df2.show(5)
display(df2)

df_customer=spark.read.schema(datastruct).csv(
    path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_customer.show(5)
display(df_customer)

In [0]:
df_usage = spark.read.csv(
    path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/",
    header=True,
    inferSchema=True,sep='\t')
df_usage.show(5)


In [0]:
usage_schema="customer_id int,voice_mins int,data_mb int,sms_count int"
df_usage = spark.read.schema(usage_schema).csv(
    path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/",
    header=True,
    sep='\t')
df_usage.show(5)

In [0]:

from pyspark.sql.types import (
    StructType, StructField,IntegerType, TimestampType,StringType
)
cust_schema = StructType([
    StructField("event_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("tower_id", StringType()),
    StructField("signal_strength", IntegerType()),
    StructField("timestamp", TimestampType())
])

df_towers = spark.read \
    .schema(cust_schema) \
    .option("header", True) \
    .option("recursiveFileLookup", True) \
    .option("pathGlobFilter", "*.csv") \
    .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*")

df_towers.show(5)

## Spark Write Operations using 
- csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options

##6. Write Operations (Data Conversion/Schema migration) – CSV Format Usecases
1. Write customer data into CSV format using overwrite mode
2. Write usage data into CSV format using append mode
3. Write tower data into CSV format with header enabled and custom separator (|)
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:

#Write customer data into CSV format using overwrite mode

df_cust=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_cust.show()

df_cust.write.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer_rgt",header=True,sep=',',mode='overwrite')
df_cust.show()

In [0]:
df_usage=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage.show(10)

df_usage.write.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage_rgt",header=True,sep='\t',mode='append')
df_usage.show()

In [0]:
df_tower=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*",header=True,sep='|',inferSchema=True)
df_tower.show()
df_tower.write.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_rgt/region*",header=True,sep='|')
df_tower.show()

In [0]:
#Read the tower data in a dataframe and show only 5 rows.
df1_tower1=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_rgt/region*",header=True,sep='|',inferSchema=True)
df1_tower1.show()

##7. Write Operations (Data Conversion/Schema migration)– JSON Format Usecases
1. Write customer data into JSON format using overwrite mode
2. Write usage data into JSON format using append mode and snappy compression format
3. Write tower data into JSON format using ignore mode and observe the behavior of this mode
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/json_targetdata")
#dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/json_targetdata")

#Write customer data into JSON format using overwrite mode

df_cust1=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_cust1.show()

df_cust1.write.json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/json_targetdata",mode='overwrite')
df_cust1.show(10)

In [0]:

#Write usage data into JSON format using append mode and snappy compression format

df_usage1=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage1.show(10)

df_usage1.write.json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/json_targetdata",mode="append",compression='snappy')
df_usage1.show(10)

In [0]:
#Write tower data into JSON format using ignore mode and observe the behavior of this mode
df_tower=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*",header=True,sep='|',inferSchema=True)
df_tower.show(5)

df_tower.write.json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/json_targetdata",header=True,sep='|',mode="ignore")
df_tower.show(5)

##8. Write Operations (Data Conversion/Schema migration) – Parquet Format Usecases
1. Write customer data into Parquet format using overwrite mode and in a gzip format
2. Write usage data into Parquet format using error mode
3. Write tower data into Parquet format with gzip compression option
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:

#Write customer data into Parquet format using overwrite mode and in a gzip format
df_cust_parquet=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_cust_parquet.show()

df_cust_parquet.write.parquet("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/parquet_targetdata",mode='overwrite',compression='gzip')
df_cust_parquet.show(10)



In [0]:
#Write usage data into Parquet format using error mode

df_usage_parquet=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage_parquet.show(10)

df_usage_parquet.write.parquet("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/parquet_targetdata",mode="error")
df_usage_parquet.show(10)



In [0]:
#Write tower data into Parquet format with gzip compression option

df_tower_parquet=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*",header=True,sep='|',inferSchema=True)
df_tower_parquet.show(5)

df_tower_parquet.write.parquet("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/parquet_targetdata",compression='gzip')
df_tower_parquet.show(5)

##9. Write Operations (Data Conversion/Schema migration) – Orc Format Usecases
1. Write customer data into ORC format using overwrite mode
2. Write usage data into ORC format using append mode
3. Write tower data into ORC format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
df_cust_orc=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_cust_orc.show()

df_cust_orc.write.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/orc_targetdata",mode='overwrite')
df_cust_orc.show(10)

In [0]:
#Write usage data into ORC format using append mode

df_usage_orc=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage_orc.show(10)

df_usage_orc.write.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/orc_targetdata",mode="append")
df_usage_orc.show(10)


In [0]:
#Write tower data into ORC format and see the output file structure
df_tower_orc=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*",header=True,sep='|',inferSchema=True)
df_tower_orc.show(5)

df_tower_orc.write.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/orc_targetdata")
df_tower_orc.show(5)

##10. Write Operations (Data Conversion/Schema migration) – Delta Format Usecases
1. Write customer data into Delta format using overwrite mode
2. Write usage data into Delta format using append mode
3. Write tower data into Delta format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
6. Compare the parquet location and delta location and try to understand what is the differentiating factor, as both are parquet files only.

In [0]:

#Write customer data into Delta format using overwrite mode

df_cust_delta=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_cust_delta.show()


df_cust_delta.write.format("delta").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/delta_targetdata",mode='overwrite')
df_cust_delta.show(10)

spark.read.format("delta").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/delta_targetdata").show(5)

In [0]:
#Write usage data into Delta format using append mode


df_usage_delta=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage_delta.show(10)

df_usage_delta.write.format("delta").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/delta_targetdata",mode="append")
df_usage_delta.show(10)

spark.read.format("delta").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/delta_targetdata").show(5)

In [0]:

#Write tower data into Delta format and see the output file structure

df_tower_delta=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region*",header=True,sep='|',inferSchema=True)
df_tower_delta.show(5)

df_tower_delta.write.format("delta").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/delta_targetdata")
df_tower_delta.show(5)

spark.read.format("delta").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/delta_targetdata").show(5)

##11. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using saveAsTable() as a managed table
2. Write usage data using saveAsTable() with overwrite mode
3. Drop the managed table and verify data removal
4. Go and check the table overview and realize it is in delta format in the Catalog.
5. Use spark.read.sql to write some simple queries on the above tables created.


In [0]:
df_cust_delta=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_cust_delta.show()

df_cust_delta.write.saveAsTable("telecom_catalog_assign.default.customertbl",mode='overwrite')
spark.read.table("telecom_catalog_assign.default.customertbl").show(6)

In [0]:
#Write usage data using saveAsTable() with overwrite mode
df_usage_delta=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage_delta.show(10)

df_usage_delta.write.saveAsTable("telecom_catalog_assign.default.usagetbl",mode='overwrite')
spark.read.table("telecom_catalog_assign.default.usagetbl").show(6)

In [0]:
spark.sql("DROP TABLE IF EXISTS telecom_catalog_assign.default.customertbl")
spark.sql("DROP TABLE IF EXISTS telecom_catalog_assign.default.usagetbl")

spark.read.table("telecom_catalog_assign.default.customertbl").show(6)
spark.read.table("telecom_catalog_assign.default.usagetbl").show(6)

##12. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using insertInto() in a new table and find the behavior
2. Write usage data using insertTable() with overwrite mode

In [0]:
#Write customer data using insertInto() in a new table and find the behavior

df_cust=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer").toDF("customer_id","Name","age","location","plan_type")
df_cust.show()
df_cust.write.insertInto("telecom_catalog_assign.default.customertbl")
spark.read.table("telecom_catalog_assign.default.customertbl").show(6)

In [0]:

#Write usage data using insertTable() with overwrite mode

df_usage=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage",header=True,sep='\t')
df_usage.show(10)

df_usage.write.insertInto("telecom_catalog_assign.default.usagetbl",overwrite=True)
spark.read.table("telecom_catalog_assign.default.usagetbl").show(6)

##13. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data into XML format using rowTag as cust
2. Write usage data into XML format using overwrite mode with the rowTag as usage
3. Download the xml data and open the file in notepad++ and see how the xml file looks like.

In [0]:

#Write customer data into XML format using rowTag as cust

df_cust.write.xml("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/xml_targetdata",rowTag='customer')
spark.read.xml("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/xml_targetdata",rowTag='customer').show(5)

In [0]:
    #Write usage data into XML format using overwrite mode with the rowTag as usage

df_usage.write.xml("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/xml_targetdata",rowTag='usage',mode='overwrite')
spark.read.xml("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/xml_targetdata",rowTag='usage').show(6)

##14. Compare all the downloaded files (csv, json, orc, parquet, delta and xml) 
1. Capture the size occupied between all of these file formats and list the formats below based on the order of size from small to big.

##15. Do a final exercise of defining one/two liner of... 
1. When to use/benifits csv
2. When to use/benifits json
3. When to use/benifit orc
4. When to use/benifit parquet
5. When to use/benifit delta
6. When to use/benifit xml
7. When to use/benifit delta tables
